# ComfyUI Distributed — Colab worker

Run this notebook **once per Google account**. It turns the Colab GPU into a
worker for your master ComfyUI, then registers itself so the adaptive video
planner can hand it one segment of the video.

```
 Colab A ─┐
 Colab B ─┼─ ComfyUI workers (tunnel each) ─→ Master ComfyUI ─→ N segments
 Colab C ─┘                                       │
                                          Adaptive Studio (plan + keyframes)
```

**Before you start**
1. The master runs ComfyUI with `--enable-cors-header`.
2. The master has this extension installed (it owns `gpu_config.json`).
3. Pick a unique `WORKER_NAME` per account (`colab-a`, `colab-b`, …).
4. If the master is reachable from the internet, put its URL in `MASTER_URL` so
   this notebook can register itself. Otherwise register manually (step 7).

Everything below is idempotent: re-running the notebook restarts the worker and
updates the same registration.

In [ ]:
# ---- parameters -----------------------------------------------------------------
WORKER_NAME  = "colab-a"          # unique per Google account
WORKER_ID    = "colab-" + WORKER_NAME.split("-")[-1]
MASTER_URL   = ""                 # e.g. "https://your-master.trycloudflare.com" (optional)
WORKER_PORT  = 8188
EXTRA_ARGS   = "--enable-cors-header --disable-auto-launch --preview-method none"
COMFY_DIR    = "/content/ComfyUI"
EXT_DIR      = COMFY_DIR + "/custom_nodes/ComfyUI-Distributed"
REPO_URL     = "https://github.com/sunlightcorondev-wq/ComfyUI-Distributed"
AUTO_REGISTER = True              # needs MASTER_URL

In [ ]:
# ---- 1. check the GPU and the runtime -------------------------------------------
import os, shutil, subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no NVIDIA GPU found")
print("python", sys.version.split()[0])
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"free disk: {free_gb:.1f} GB")
assert free_gb > 20, "Colab needs ~20 GB free for ComfyUI and models"

In [ ]:
# ---- 2. install ComfyUI + this extension ----------------------------------------
# Colab ships a matching torch, so only the non-torch requirements are installed.
if not os.path.isdir(COMFY_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}

req = "/tmp/comfy-requirements.txt"
!grep -viE "^(torch|torchvision|torchaudio)([=<> ]|$)" {COMFY_DIR}/requirements.txt > {req} || true
!pip install -q -r {req}

if not os.path.isdir(EXT_DIR):
    !git clone --depth 1 {REPO_URL} {EXT_DIR}
else:
    !git -C {EXT_DIR} pull --ff-only

print("installed")

In [ ]:
# ---- 3. start the worker --------------------------------------------------------
import json, pathlib, signal, time

LOG = pathlib.Path("/content/worker.log")
PID = pathlib.Path("/content/worker.pid")

def stop_worker():
    if PID.exists():
        try:
            os.kill(int(PID.read_text().strip()), signal.SIGTERM)
            time.sleep(2)
        except (ProcessLookupError, ValueError):
            pass
        PID.unlink(missing_ok=True)

def start_worker():
    stop_worker()
    env = dict(os.environ, COMFYUI_IS_WORKER="1", PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "main.py", "--listen", "--port", str(WORKER_PORT)] + EXTRA_ARGS.split()
    log = open(LOG, "w")
    process = subprocess.Popen(cmd, cwd=COMFY_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    PID.write_text(str(process.pid))
    print(f"worker pid {process.pid}, logging to {LOG}")
    return process

def tail(lines=15):
    content = LOG.read_text(errors="ignore").splitlines() if LOG.exists() else []
    print("\n".join(content[-lines:]) or "(no log yet)")

start_worker()
time.sleep(20)
tail()

In [ ]:
# ---- 4. expose the worker with a Cloudflare quick tunnel ------------------------
import re, pathlib, subprocess, time

CLOUDFLARED = "/usr/local/bin/cloudflared"
TUNNEL_LOG = pathlib.Path("/content/tunnel.log")

if not os.path.exists(CLOUDFLARED):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CLOUDFLARED}
    !chmod +x {CLOUDFLARED}

def tunnel_running():
    return subprocess.run(["pgrep", "-f", "cloudflared tunnel"], capture_output=True).returncode == 0

if not tunnel_running():
    with open(TUNNEL_LOG, "w") as log:
        subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url",
                          f"http://127.0.0.1:{WORKER_PORT}"], stdout=log, stderr=subprocess.STDOUT)

TUNNEL_URL = ""
for _ in range(60):
    text = TUNNEL_LOG.read_text(errors="ignore") if TUNNEL_LOG.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        TUNNEL_URL = match.group(0)
        break
    time.sleep(2)

print("public worker URL:", TUNNEL_URL or "not ready yet — re-run this cell")

In [ ]:
# ---- 5. register this account with the master -----------------------------------
# The master stores workers in gpu_config.json; this posts the same fields the
# Distributed panel writes when you add a Remote worker.
import json, urllib.request

def post_json(url, payload):
    request = urllib.request.Request(
        url, data=json.dumps(payload).encode(), method="POST",
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=20) as response:
        return json.loads(response.read() or b"{}")

if AUTO_REGISTER and MASTER_URL and TUNNEL_URL:
    result = post_json(f"{MASTER_URL.rstrip('/')}/distributed/config/update_worker", {
        "worker_id": WORKER_ID,
        "name": WORKER_NAME,
        "type": "remote",
        "host": TUNNEL_URL,      # explicit scheme: port is taken from the URL
        "port": 443,
        "cuda_device": 0,
        "enabled": True,
    })
    print("registered:", result)
elif AUTO_REGISTER:
    print("Set MASTER_URL and re-run cell 4 + 5, or add the worker manually:")
    print(f"  panel -> Add Worker -> Remote -> host {TUNNEL_URL or '<tunnel url>'} port 443")
else:
    print("AUTO_REGISTER is off; add the worker manually in the Distributed panel")

In [ ]:
# ---- 6. verify ------------------------------------------------------------------
import json, urllib.request

def get_json(url):
    with urllib.request.urlopen(url, timeout=20) as response:
        return json.loads(response.read() or b"{}")

print("local worker:", get_json(f"http://127.0.0.1:{WORKER_PORT}/system_stats")["system"]["comfyui_version"])

if MASTER_URL:
    config = get_json(f"{MASTER_URL.rstrip('/')}/distributed/config")
    ids = [worker.get("id") for worker in config.get("workers", [])]
    print("master sees:", ids)
    print("this account registered:" , WORKER_ID in ids)

## How this account gets a video segment

Open **Adaptive Studio** on the master (`python3 apps/adaptive_studio/server.py`)
or use the **Adaptive Video Segments** node. The plan counts enabled workers at
dispatch time, so this account receives exactly one segment:

| | |
|---|---|
| 3 accounts | 3 segments, ~2 s each |
| 6 accounts | 6 segments, ~1 s each |
| adding this account | one more segment, same saved workflow |

`worker_N` in the plan is the Nth enabled worker in the panel, so keep the same
order you want the final video stitched in.

## Troubleshooting

| Symptom | Fix |
|---|---|
| Master shows the worker as offline | Re-run cell 4; quick tunnels get a new URL on restart, then re-register (cell 5) |
| Tunnel URL never appears | Re-run cell 4; check `/content/tunnel.log` |
| `CUDA out of memory` | Lower resolution/frames per segment, or use fewer accounts for 720p |
| Worker disappears after ~2 h | Colab recycles sessions; re-run cells 3–5 |
| Master cannot reach the worker | The master needs outbound internet access to `*.trycloudflare.com` |